# Qwen3.5-4B whole-system RotQuant co-design

This notebook searches model-weight precision, rotation-aware K/V-cache
precision, and optional post-quantization recovery as one staged system.
It measures the Pareto frontier over complete model size, prompt-dependent
cache size, WikiText-2 perplexity, held-out cache KL/NLL, free-running
trajectory agreement, and direct K/V reconstruction NMSE.

## Goal

Find the smallest deployable Qwen3.5-4B configuration that keeps mean
perplexity degradation near 5%, worst-seed degradation below 10%, and
cache divergence no worse than the matched uniform W4/K4/V4 control.

“Joint” here means staged whole-system co-design: screen weight recipes,
cross the viable weights with cache recipes, recover the best joint
candidates, then validate the final combination. It is deliberately not
an intractable Cartesian search or an unsupported differentiable claim.

### Key assumptions

- C4 calibration/selection calls are disjoint from held-out cache calls,
  trajectory prompts, and WikiText-2 evaluation.
- Dynamic precisions are static model-specific deployment recipes; they do
  not change per token at inference time.
- CUDA `fallback=true` materializes source-dtype weights for quality work.
  Logical packed bytes are valid; fallback VRAM and throughput are not.
- The vision tower, tied vocabulary, and excluded recurrent gate projections
  stay in source precision and are included in estimated complete-model size.
- Only full-attention K/V is quantized. Qwen recurrent linear-attention state
  is retained unchanged and included in total cache accounting.
- Every run is content-addressed and persisted to Drive. Interrupted runs
  resume without repeating completed configurations.

## Trial ladder

| Stage | Search |
|---|---|
| Controls | Source weights and uniform W4/K4/V4 |
| A: weights | Uniform W3/W4 plus dynamic 2/3/4/8-bit weight budgets |
| B: joint | Best weight recipes × asymmetric, codebook, and dynamic K/V profiles |
| C: recovery | Butterfly/scale block recovery and rank-4 LoRA-QAT on the best joint base |
| D: release | Two finalists across seeds 0/1/2 |
| E: context | Winner and source-weight control at 1,024-token prefill |

With defaults this is roughly fifty-five resumable Qwen runs. Change the flags
below to make a shorter pilot or a broader release study.

## Setup

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/CodeHalwell/rotquant.git"
REPO_REF = "main"
REPO_DIR = Path("/content/rotquant-joint-matrix")
MODEL_ID = "unsloth/Qwen3.5-4B"
CONFIG_RELATIVE_PATH = Path("configs/qwen35_4b_joint_cuda.yaml")

USE_GOOGLE_DRIVE = True
DRIVE_RESULT_ROOT = Path("/content/drive/MyDrive/rotquant/qwen35_joint_matrix")
DRIVE_KV_RESULT_ROOT = Path("/content/drive/MyDrive/rotquant/qwen35_kv_matrix")
LOCAL_RESULT_ROOT = Path("/content/qwen35_joint_matrix")
FROZEN_KV_TRANSFER_RUN = "49d3cf1182f0"

EVAL_SEQ_LEN = 256
EVAL_MAX_SAMPLES = 64
MEAN_PPL_GATE = 0.05
WORST_PPL_GATE = 0.10
KV_KL_GATE_MULTIPLIER = 1.05
KV_KL_SCORE_WEIGHT = 0.10
N_WEIGHT_FINALISTS = 3
N_JOINT_FINALISTS = 2

CONFIRM_EXPENSIVE_RUN = False
FORCE_RERUN = False
RUN_WEIGHT_SCREEN = True
RUN_JOINT_MATRIX = True
RUN_BLOCK_RECOVERY = True
RUN_LORA_RECOVERY = True
RUN_SEED_VALIDATION = True
RUN_LONG_CONTEXT_CONFIRMATION = True
DOWNLOAD_RESULTS = True

print({
    "repo_ref": REPO_REF,
    "weight_finalists": N_WEIGHT_FINALISTS,
    "joint_finalists": N_JOINT_FINALISTS,
    "confirm_expensive_run": CONFIRM_EXPENSIVE_RUN,
})

### 1. Verify the CUDA runtime

In [ ]:
import os
import subprocess
import sys
import torch

assert torch.cuda.is_available(), "Select a CUDA GPU runtime before continuing."
gpu = torch.cuda.get_device_properties(0)
vram_gib = gpu.total_memory / 2**30
print(f"GPU: {gpu.name} | VRAM: {vram_gib:.1f} GiB")
print(f"torch={torch.__version__} | CUDA={torch.version.cuda}")
if vram_gib < 40:
    print("WARNING: cached fallback may OOM below 40 GiB.")
subprocess.run(["nvidia-smi"], check=True)

### 2. Mount Drive and fetch the latest revision

In [ ]:
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULT_BASE = DRIVE_RESULT_ROOT
    FROZEN_KV_SUMMARY_PATH = DRIVE_KV_RESULT_ROOT / "frozen_transfer" / FROZEN_KV_TRANSFER_RUN / "kv_frozen_transfer_summary.json"
else:
    RESULT_BASE = LOCAL_RESULT_ROOT
    FROZEN_KV_SUMMARY_PATH = LOCAL_RESULT_ROOT / "frozen_transfer_summary.json"
RESULT_BASE.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run([
        "git", "clone", "--branch", REPO_REF, "--single-branch",
        REPO_URL, str(REPO_DIR),
    ], check=True)
else:
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", REPO_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", REPO_REF], cwd=REPO_DIR, check=True)

commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
).strip()
RESULT_ROOT = RESULT_BASE / commit[:12]
RUN_ROOT = RESULT_ROOT / "runs"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Using commit {commit}; results: {RESULT_ROOT}")

### 3. Install dependencies without replacing CUDA PyTorch

In [ ]:
runtime_packages = [
    "transformers==5.9.0", "datasets>=4.8", "accelerate",
    "safetensors", "sentencepiece", "scipy", "pyyaml",
    "pandas", "matplotlib", "huggingface_hub",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", *runtime_packages],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR), "--no-deps"],
    check=True,
)

pil_probe_command = [
    sys.executable, "-c",
    "from PIL import Image, ImageColor, ImageDraw, ImageFont, ImageText; print(Image.__version__)",
]
pil_probe = subprocess.run(pil_probe_command, capture_output=True, text=True)
if pil_probe.returncode != 0:
    print("Detected an inconsistent live Pillow installation; repairing it once.")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "--force-reinstall",
        "--no-cache-dir", "pillow==12.2.0",
    ], check=True)
    subprocess.run(pil_probe_command, check=True)
    raise RuntimeError(
        "Pillow was repaired. Use Runtime > Restart session, then rerun from the top."
    )

repo_path = str(REPO_DIR)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
os.environ["TORCH_ALLOW_TF32_CUBLAS_OVERRIDE"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")
print(f"Runtime ready; Pillow={pil_probe.stdout.strip()}.")

### 4. Validate the repository contract

In [ ]:
import json
import yaml

config_path = REPO_DIR / CONFIG_RELATIVE_PATH
required_paths = [
    config_path,
    REPO_DIR / "rotquant/dynamic.py",
    REPO_DIR / "rotquant/eval/kv_cache.py",
    REPO_DIR / "scripts/run_experiment.py",
]
missing = [str(path) for path in required_paths if not path.exists()]
assert not missing, "Missing required files: " + ", ".join(missing)
with config_path.open() as handle:
    base_config = yaml.safe_load(handle)
assert base_config["model"] == MODEL_ID
assert base_config["patch"]["fallback"] is True
assert base_config["eval"]["kv_cache"]["eval_offset_batches"] == 4
kv_source = (REPO_DIR / "rotquant/eval/kv_cache.py").read_text()
assert "prefill_key_nmse" in kv_source
assert "eval_offset_batches" in kv_source
assert "frozen_recipe" in kv_source
assert FROZEN_KV_SUMMARY_PATH.exists(), f"Missing frozen K/V result: {FROZEN_KV_SUMMARY_PATH}"
with FROZEN_KV_SUMMARY_PATH.open() as handle:
    frozen_kv_summary = json.load(handle)
assert frozen_kv_summary["recommendation"] == "mixed"
assert frozen_kv_summary["source_run_id"] == "bf94f2045a90"
FROZEN_MIXED_KV_RECIPE = frozen_kv_summary["mixed_recipe"]
assert FROZEN_MIXED_KV_RECIPE and all({"layer", "key_bits", "value_bits"} <= set(row) for row in FROZEN_MIXED_KV_RECIPE)
print(yaml.safe_dump(base_config, sort_keys=False))

## Steps

### 5. Define weight, cache, and recovery profiles

In [ ]:
def encoded_override(path, value):
    return f"{path}={json.dumps(value, separators=(',', ':'))}"

def dynamic_weight(target_bpw, *, guarded=False):
    rules = []
    if guarded:
        rules = [
            {"match": "self_attn.o_proj", "min_bits": 4},
            {"match": "linear_attn.out_proj", "min_bits": 4},
            {"match": "mlp.down_proj", "min_bits": 4},
        ]
    settings = {
        "candidate_bits": [2, 3, 4, 8],
        "target_bpw": target_bpw,
        "max_tokens": 64,
        "local_weight": 1.0,
        "global_kl_weight": 0.0,
        "global_kl_batches": 0,
        "rules": rules,
    }
    return ["patch.enabled=true", "quant.bits=4", encoded_override("patch.dynamic", settings)]

def uniform_weight(bits):
    return [
        "patch.enabled=true", f"quant.bits={bits}",
        encoded_override("patch.dynamic", None),
    ]

def uniform_kv(key_bits, value_bits, *, codebook="gaussian", group_size=64):
    return [
        encoded_override("eval.kv_cache.dynamic", None),
        encoded_override("eval.kv_cache.frozen_recipe", None),
        f"eval.kv_cache.key_bits={key_bits}",
        f"eval.kv_cache.value_bits={value_bits}",
        f"eval.kv_cache.codebook={codebook}",
        f"eval.kv_cache.group_size={group_size}",
    ]

def dynamic_kv(target_bpv, *, codebook="gaussian"):
    settings = {
        "candidate_bits": [2, 3, 4, 8],
        "target_bpv": target_bpv,
        "selection_batches": 4,
    }
    return [
        "eval.kv_cache.key_bits=null",
        "eval.kv_cache.value_bits=null",
        f"eval.kv_cache.codebook={codebook}",
        encoded_override("eval.kv_cache.frozen_recipe", None),
        encoded_override("eval.kv_cache.dynamic", settings),
    ]

def frozen_kv(recipe, *, codebook="gaussian"):
    return [
        "eval.kv_cache.key_bits=null",
        "eval.kv_cache.value_bits=null",
        f"eval.kv_cache.codebook={codebook}",
        encoded_override("eval.kv_cache.dynamic", None),
        encoded_override("eval.kv_cache.frozen_recipe", recipe),
    ]

COMMON_EVAL = [
    "eval.perplexity=true",
    f"eval.ppl.seq_len={EVAL_SEQ_LEN}",
    f"eval.ppl.max_samples={EVAL_MAX_SAMPLES}",
    "eval.kv_cache.batches=4",
    "eval.kv_cache.eval_offset_batches=4",
]
WEIGHT_SCREEN_KV = uniform_kv(4, 4)

WEIGHT_PROFILES = {
    "uniform_w4": uniform_weight(4),
    "uniform_w3": uniform_weight(3),
    "dynamic_w2.75": dynamic_weight(2.75),
    "dynamic_w3.25": dynamic_weight(3.25),
    "dynamic_w3.625": dynamic_weight(3.625),
    "dynamic_w4.125": dynamic_weight(4.125),
    "dynamic_w3.25_guarded": dynamic_weight(3.25, guarded=True),
}

KV_PROFILES = {
    "uniform_k2_v3": uniform_kv(2, 3),
    "uniform_k2_v4": uniform_kv(2, 4),
    "uniform_k3_v3": uniform_kv(3, 3),
    "uniform_k3_v4": uniform_kv(3, 4),
    "uniform_k4_v4": uniform_kv(4, 4),
    "uniform_k4_v8": uniform_kv(4, 8),
    "uniform_codebook_k4_v4": uniform_kv(4, 4, codebook="uniform"),
    "dynamic_g_2.25": dynamic_kv(2.25),
    "dynamic_g_3.25": dynamic_kv(3.25),
    "dynamic_g_4.25": dynamic_kv(4.25),
    "dynamic_g_5.25": dynamic_kv(5.25),
    "dynamic_u_3.25": dynamic_kv(3.25, codebook="uniform"),
    "dynamic_u_4.25": dynamic_kv(4.25, codebook="uniform"),
    "frozen_mixed_3.25": frozen_kv(FROZEN_MIXED_KV_RECIPE),
}

block_settings = {
    "objective": "block", "steps": 12, "lr": 0.0015,
    "train_batches": 4, "validation_batches": 2, "selection_batches": 2,
    "learn_scales": True, "scale_lr": 0.01,
    "scale_multiplier_min": 0.5, "scale_multiplier_max": 1.5,
    "propagate_quantized_inputs": True, "max_grad_norm": 1.0,
    "restore_best": True, "early_stopping_patience": 4,
    "validation_min_improvement": 0.001,
    "selection_min_improvement": 0.005,
    "distill_steps": 0,
}
lora_settings = dict(block_settings)
lora_settings.update({
    "distill_steps": 24, "distill_lr": 0.0002,
    "distill_scale_lr": 0.005,
    "distill_train_batches": 8,
    "distill_validation_batches": 2,
    "distill_selection_batches": 4,
    "distill_temperature": 2.0,
    "distill_kl_weight": 1.0, "distill_ce_weight": 0.1,
    "distill_angle_l2": 0.0001,
    "distill_max_grad_norm": 1.0,
    "distill_early_stopping_patience": 6,
    "distill_validation_min_improvement": 0.0,
    "distill_selection_min_improvement": 0.001,
    "distill_lora_rank": 4, "distill_lora_alpha": 8.0,
    "distill_lora_lr": 0.0005,
    "distill_train_rotations": False,
    "distill_train_scales": False,
})
RECOVERY_PROFILES = {
    "block_scale": [
        "patch.rotation=butterfly",
        encoded_override("patch.train_rotation", block_settings),
    ],
    "lora_qat": [
        "patch.rotation=butterfly",
        encoded_override("patch.train_rotation", lora_settings),
    ],
}

estimated_runs = (
    1 + len(WEIGHT_PROFILES)
    + N_WEIGHT_FINALISTS * len(KV_PROFILES)
    + int(RUN_BLOCK_RECOVERY) + int(RUN_LORA_RECOVERY)
    + 2 * N_JOINT_FINALISTS
    + 2 * int(RUN_LONG_CONTEXT_CONFIRMATION)
)
print({
    "weight_profiles": len(WEIGHT_PROFILES),
    "kv_profiles": len(KV_PROFILES),
    "estimated_max_runs": estimated_runs,
})

### 6. Define the content-addressed experiment runner

In [ ]:
import hashlib
import shlex
from typing import Iterable

PROTOCOL_VERSION = "joint-weight-kv-v1"
trial_records = {}
trial_registry = {}
registry_path = RESULT_ROOT / "trial_registry.json"

def latest_result(output_dir):
    candidates = sorted(output_dir.glob("*.json"), key=lambda path: path.stat().st_mtime)
    if not candidates:
        return None
    with candidates[-1].open() as handle:
        return json.load(handle)

def persist_registry():
    with registry_path.open("w") as handle:
        json.dump(list(trial_registry.values()), handle, indent=2)

def run_logged(command, *, cwd, log_path):
    with log_path.open("w") as log_handle:
        process = subprocess.Popen(
            command, cwd=cwd, env=os.environ.copy(),
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="", flush=True)
            log_handle.write(line)
            log_handle.flush()
        returncode = process.wait()
    if returncode != 0:
        lines = log_path.read_text(errors="replace").splitlines()
        tail = "\n".join(lines[-120:])
        raise RuntimeError(
            f"Trial subprocess failed with exit code {returncode}. "
            f"Full log: {log_path}\n--- log tail ---\n{tail}"
        )

def run_trial(stage, trial_name, overrides: Iterable[str], *, seed=0, weight=None, kv=None):
    overrides = list(overrides)
    signature = hashlib.sha256(json.dumps({
        "protocol": PROTOCOL_VERSION,
        "commit": commit,
        "seed": seed,
        "overrides": overrides,
    }, sort_keys=True).encode()).hexdigest()[:12]
    output_dir = RUN_ROOT / signature
    output_dir.mkdir(parents=True, exist_ok=True)
    payload = None if FORCE_RERUN else latest_result(output_dir)
    if payload is None:
        command = [
            sys.executable,
            str(REPO_DIR / "scripts/run_experiment.py"),
            str(config_path),
            "--output-dir", str(output_dir),
            "--seed", str(seed),
        ]
        for override in overrides:
            command.extend(["--set", override])
        print("Running:", shlex.join(command), flush=True)
        run_logged(command, cwd=REPO_DIR, log_path=output_dir / "subprocess.log")
        payload = latest_result(output_dir)
        assert payload is not None, f"No result JSON was written to {output_dir}"
    else:
        print(f"Reusing {trial_name}: {signature}")

    key = f"{stage}:{trial_name}:s{seed}"
    trial_records[key] = payload
    trial_registry[key] = {
        "key": key, "stage": stage, "trial": trial_name,
        "weight": weight, "kv": kv, "seed": seed,
        "signature": signature, "overrides": overrides,
        "result_dir": str(output_dir),
    }
    persist_registry()
    return payload

def combined_overrides(weight_name, kv_name, *extra):
    return [
        *WEIGHT_PROFILES[weight_name],
        *KV_PROFILES[kv_name],
        *COMMON_EVAL,
        *extra,
    ]

### 7. Confirm the complete run

In [ ]:
assert CONFIRM_EXPENSIVE_RUN, (
    "Inspect the trial ladder, then set CONFIRM_EXPENSIVE_RUN=True."
)

### 8. Establish source and weight-screen controls

In [ ]:
source_overrides = ["patch.enabled=false", *WEIGHT_SCREEN_KV, *COMMON_EVAL]
source_result = run_trial(
    "controls", "source_w16_k4_v4", source_overrides,
    seed=0, weight="source_w16", kv="uniform_k4_v4",
)
source_ppl = source_result["metrics"]["ppl_wikitext2"]
print(f"Source PPL: {source_ppl:.4f}")

weight_screen_results = {}
if RUN_WEIGHT_SCREEN:
    for weight_name, weight_overrides in WEIGHT_PROFILES.items():
        overrides = [*weight_overrides, *WEIGHT_SCREEN_KV, *COMMON_EVAL]
        weight_screen_results[weight_name] = run_trial(
            "weight_screen", f"{weight_name}__uniform_k4_v4", overrides,
            seed=0, weight=weight_name, kv="uniform_k4_v4",
        )
else:
    raise RuntimeError("Weight screening is required for the staged joint search.")

### 9. Build comparable size and quality rows

In [ ]:
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download

index_path = hf_hub_download(MODEL_ID, "model.safetensors.index.json")
with open(index_path) as handle:
    source_weight_bytes = json.load(handle)["metadata"]["total_size"]

def estimated_complete_weight_bytes(payload):
    metrics = payload["metrics"]
    if "fp16_weight_bytes" not in metrics:
        return source_weight_bytes
    deployed = metrics.get(
        "packed_plus_auxiliary_bytes", metrics["packed_weight_bytes"]
    )
    return source_weight_bytes - metrics["fp16_weight_bytes"] + deployed

def report_row(stage, trial, payload, *, weight, kv, seed):
    metrics = payload["metrics"]
    cache = metrics.get("kv_cache", {})
    trajectory = metrics.get("trajectory", {})
    weight_bytes = estimated_complete_weight_bytes(payload)
    cache_bytes = float(cache.get("deployed_total_cache_bytes", 0.0))
    dynamic_weight = metrics.get("dynamic_quantization", {})
    dynamic_cache = cache.get("dynamic", {})
    ppl = float(metrics.get("ppl_wikitext2", np.nan))
    return {
        "stage": stage, "trial": trial, "weight": weight, "kv": kv, "seed": seed,
        "ppl": ppl,
        "relative_ppl": ppl / source_ppl - 1.0 if np.isfinite(ppl) else np.nan,
        "weight_bpw": metrics.get(
            "effective_bits_per_weight", metrics.get("bits_per_weight_mean", 16.0)
        ),
        "estimated_weight_GB": weight_bytes / 1e9,
        "weight_reduction": 1.0 - weight_bytes / source_weight_bytes,
        "cache_kl": cache.get("mean_teacher_kl", np.nan),
        "cache_nll_delta": cache.get("nll_delta", np.nan),
        "cache_top1": cache.get("top1_agreement", np.nan),
        "cache_cosine": cache.get("mean_logit_cosine", np.nan),
        "key_bits": cache.get("key_bits", np.nan),
        "value_bits": cache.get("value_bits", np.nan),
        "effective_kv_bpv": cache.get("effective_kv_bpv", np.nan),
        "prefill_key_nmse": cache.get("prefill_key_nmse", np.nan),
        "prefill_value_nmse": cache.get("prefill_value_nmse", np.nan),
        "deployed_cache_MB": cache_bytes / 1e6,
        "total_system_GB_at_prompt": (weight_bytes + cache_bytes) / 1e9,
        "trajectory_token_agreement": trajectory.get("token_agreement", np.nan),
        "trajectory_exact_rate": trajectory.get("exact_trajectory_rate", np.nan),
        "weight_target_reached": dynamic_weight.get("target_reached", True),
        "cache_target_reached": dynamic_cache.get("target_reached", True),
        "weight_counts": json.dumps(dynamic_weight.get("counts_by_bits", {}), sort_keys=True),
        "cache_key_counts": json.dumps(dynamic_cache.get("key_counts_by_bits", {}), sort_keys=True),
        "cache_value_counts": json.dumps(dynamic_cache.get("value_counts_by_bits", {}), sort_keys=True),
        "adapter_MB": metrics.get("adapter_parameter_bytes", 0) / 1e6,
    }

source_row = report_row(
    "controls", "source_w16_k4_v4", source_result,
    weight="source_w16", kv="uniform_k4_v4", seed=0,
)
weight_rows = [
    report_row(
        "weight_screen", f"{name}__uniform_k4_v4", payload,
        weight=name, kv="uniform_k4_v4", seed=0,
    )
    for name, payload in weight_screen_results.items()
]
weight_screen = pd.DataFrame([source_row, *weight_rows])
display(weight_screen.style.format({
    "ppl": "{:.4f}", "relative_ppl": "{:+.2%}",
    "weight_bpw": "{:.3f}", "estimated_weight_GB": "{:.3f}",
    "weight_reduction": "{:.2%}", "cache_kl": "{:.4g}",
    "cache_top1": "{:.3f}", "prefill_key_nmse": "{:.3g}",
    "prefill_value_nmse": "{:.3g}",
}))

### 10. Select non-dominated weight finalists

In [ ]:
def pareto_mask(frame, columns):
    rows = frame.reset_index(drop=True)
    keep = []
    for _, row in rows.iterrows():
        no_worse = np.ones(len(rows), dtype=bool)
        strictly_better = np.zeros(len(rows), dtype=bool)
        for column in columns:
            no_worse &= rows[column].to_numpy() <= row[column]
            strictly_better |= rows[column].to_numpy() < row[column]
        keep.append(not np.any(no_worse & strictly_better))
    return np.array(keep, dtype=bool)

quantized_weights = weight_screen[weight_screen["weight"] != "source_w16"].copy()
eligible_weights = quantized_weights[
    (quantized_weights["relative_ppl"] <= WORST_PPL_GATE)
    & quantized_weights["weight_target_reached"]
].copy()
assert not eligible_weights.empty, "No weight recipe passed the seed-0 10% PPL gate."
weight_pareto = eligible_weights.reset_index(drop=True)[pareto_mask(
    eligible_weights, ["estimated_weight_GB", "ppl", "cache_kl"]
)]

ordered = ["uniform_w4"]
anchors = [
    eligible_weights.sort_values("estimated_weight_GB").iloc[0]["weight"],
    eligible_weights.sort_values("ppl").iloc[0]["weight"],
]
anchors += weight_pareto.sort_values(
    ["estimated_weight_GB", "ppl"]
)["weight"].tolist()
for name in anchors:
    if name not in ordered and name in set(eligible_weights["weight"]):
        ordered.append(name)
    if len(ordered) >= N_WEIGHT_FINALISTS:
        break
weight_finalists = ordered[:N_WEIGHT_FINALISTS]
print({"weight_finalists": weight_finalists})
display(weight_pareto)

### 11. Cross weight finalists with cache profiles

In [ ]:
joint_results = {}
if RUN_JOINT_MATRIX:
    for weight_name in weight_finalists:
        for kv_name in KV_PROFILES:
            trial_name = f"{weight_name}__{kv_name}"
            overrides = combined_overrides(weight_name, kv_name)
            joint_results[(weight_name, kv_name)] = run_trial(
                "joint_matrix", trial_name, overrides,
                seed=0, weight=weight_name, kv=kv_name,
            )
else:
    raise RuntimeError("The joint matrix is required before recovery selection.")

### 12. Compute the joint Pareto frontier

In [ ]:
joint_rows = [
    report_row(
        "joint_matrix", f"{weight}__{kv}", payload,
        weight=weight, kv=kv, seed=0,
    )
    for (weight, kv), payload in joint_results.items()
]
joint_matrix = pd.DataFrame(joint_rows)
baseline = joint_matrix[
    (joint_matrix["weight"] == "uniform_w4")
    & (joint_matrix["kv"] == "uniform_k4_v4")
].iloc[0]
cache_kl_gate = baseline["cache_kl"] * KV_KL_GATE_MULTIPLIER
joint_matrix["joint_score"] = (
    joint_matrix["relative_ppl"]
    + KV_KL_SCORE_WEIGHT * joint_matrix["cache_kl"]
    + 0.01 * joint_matrix["cache_nll_delta"].clip(lower=0)
)
eligible_joint = joint_matrix[
    (joint_matrix["relative_ppl"] <= WORST_PPL_GATE)
    & (joint_matrix["cache_kl"] <= cache_kl_gate)
    & joint_matrix["weight_target_reached"]
    & joint_matrix["cache_target_reached"]
].copy()
assert not eligible_joint.empty, "No joint recipe passed the seed-0 gates."
joint_pareto = eligible_joint.reset_index(drop=True)[pareto_mask(
    eligible_joint,
    ["total_system_GB_at_prompt", "ppl", "cache_kl"],
)]

selected_rows = []
anchors = [
    eligible_joint.sort_values("total_system_GB_at_prompt").iloc[0],
    eligible_joint.sort_values("joint_score").iloc[0],
]
for _, row in joint_pareto.sort_values(
    ["total_system_GB_at_prompt", "joint_score"]
).iterrows():
    anchors.append(row)
seen = set()
for row in anchors:
    key = (row["weight"], row["kv"])
    if key not in seen:
        selected_rows.append(row)
        seen.add(key)
    if len(selected_rows) >= N_JOINT_FINALISTS:
        break

joint_finalists = []
for row in selected_rows:
    weight_name, kv_name = row["weight"], row["kv"]
    joint_finalists.append({
        "name": f"{weight_name}__{kv_name}",
        "weight": weight_name,
        "kv": kv_name,
        "overrides": combined_overrides(weight_name, kv_name),
        "payload": joint_results[(weight_name, kv_name)],
    })
print({"joint_finalists": [item["name"] for item in joint_finalists]})
display(joint_matrix.sort_values(
    ["total_system_GB_at_prompt", "joint_score"]
).style.format({
    "ppl": "{:.4f}", "relative_ppl": "{:+.2%}",
    "estimated_weight_GB": "{:.3f}",
    "deployed_cache_MB": "{:.2f}",
    "total_system_GB_at_prompt": "{:.3f}",
    "cache_kl": "{:.4g}", "cache_nll_delta": "{:+.4f}",
    "cache_top1": "{:.3f}", "effective_kv_bpv": "{:.3f}",
    "prefill_key_nmse": "{:.3g}", "prefill_value_nmse": "{:.3g}",
}))
print("Pareto trials:", joint_pareto["trial"].tolist())

### 13. Plot the weight and whole-system frontiers

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for _, row in weight_screen.iterrows():
    axes[0].scatter(row["estimated_weight_GB"], row["ppl"], s=55)
    axes[0].annotate(row["weight"], (row["estimated_weight_GB"], row["ppl"]),
                     xytext=(4, 4), textcoords="offset points", fontsize=7)
axes[0].axhline(source_ppl * (1 + MEAN_PPL_GATE), linestyle="--", color="tab:orange")
axes[0].axhline(source_ppl * (1 + WORST_PPL_GATE), linestyle=":", color="tab:red")
axes[0].set(title="Weight screen", xlabel="Estimated complete weight size (GB)", ylabel="WikiText-2 PPL")
axes[0].grid(alpha=0.25)

for weight_name, group in joint_matrix.groupby("weight"):
    axes[1].scatter(group["total_system_GB_at_prompt"], group["cache_kl"],
                    s=45, label=weight_name, alpha=0.8)
axes[1].scatter(joint_pareto["total_system_GB_at_prompt"], joint_pareto["cache_kl"],
                s=120, facecolors="none", edgecolors="black", label="Pareto")
axes[1].axhline(cache_kl_gate, linestyle="--", color="tab:red", label="KL gate")
axes[1].set(title="Joint weight + 256-token cache frontier",
            xlabel="Estimated total system state (GB)", ylabel="Held-out cache KL")
axes[1].grid(alpha=0.25)
axes[1].legend(fontsize=7)
plt.tight_layout()
frontier_plot = RESULT_ROOT / "joint_frontiers.png"
fig.savefig(frontier_plot, dpi=170)
plt.show()
print(frontier_plot)

### 14. Apply post-quantization recovery to the best joint base

In [ ]:
recovery_results = {}
recovery_candidates = []
recovery_base = joint_finalists[0]
requested_recovery = []
if RUN_BLOCK_RECOVERY:
    requested_recovery.append("block_scale")
if RUN_LORA_RECOVERY:
    requested_recovery.append("lora_qat")

for recovery_name in requested_recovery:
    trial_name = f"{recovery_base['name']}__{recovery_name}"
    overrides = [
        *WEIGHT_PROFILES[recovery_base["weight"]],
        *RECOVERY_PROFILES[recovery_name],
        *KV_PROFILES[recovery_base["kv"]],
        *COMMON_EVAL,
    ]
    payload = run_trial(
        "recovery", trial_name, overrides, seed=0,
        weight=f"{recovery_base['weight']}+{recovery_name}",
        kv=recovery_base["kv"],
    )
    recovery_results[recovery_name] = payload
    recovery_candidates.append({
        "name": trial_name,
        "weight": f"{recovery_base['weight']}+{recovery_name}",
        "kv": recovery_base["kv"],
        "overrides": overrides,
        "payload": payload,
    })

recovery_rows = [
    report_row(
        "recovery", item["name"], item["payload"],
        weight=item["weight"], kv=item["kv"], seed=0,
    )
    for item in recovery_candidates
]
recovery_table = pd.DataFrame(recovery_rows)
if not recovery_table.empty:
    display(recovery_table)
else:
    print("Recovery stage disabled.")

### 15. Select release candidates and validate seeds

In [ ]:
base_candidates = list(joint_finalists)
all_candidates = [*base_candidates, *recovery_candidates]
candidate_rows = []
for item in all_candidates:
    candidate_rows.append(report_row(
        "release_candidate", item["name"], item["payload"],
        weight=item["weight"], kv=item["kv"], seed=0,
    ))
release_table = pd.DataFrame(candidate_rows)
release_table["joint_score"] = (
    release_table["relative_ppl"]
    + KV_KL_SCORE_WEIGHT * release_table["cache_kl"]
    + 0.01 * release_table["cache_nll_delta"].clip(lower=0)
)
release_eligible = release_table[
    (release_table["relative_ppl"] <= WORST_PPL_GATE)
    & (release_table["cache_kl"] <= cache_kl_gate)
    & release_table["weight_target_reached"]
    & release_table["cache_target_reached"]
].sort_values(["total_system_GB_at_prompt", "joint_score"])
assert not release_eligible.empty, "No release candidate passed the gates."
release_names = release_eligible["trial"].head(N_JOINT_FINALISTS).tolist()
release_finalists = [item for item in all_candidates if item["name"] in release_names]
print({"release_finalists": release_names})
display(release_table)

validation_records = []
for item in release_finalists:
    validation_records.append((item, 0, item["payload"]))
if RUN_SEED_VALIDATION:
    for item in release_finalists:
        for seed in (1, 2):
            payload = run_trial(
                "seed_validation", item["name"], item["overrides"],
                seed=seed, weight=item["weight"], kv=item["kv"],
            )
            validation_records.append((item, seed, payload))
else:
    print("Seed validation disabled; do not treat seed-0 ranking as final.")

validation_rows = [
    report_row(
        "seed_validation", item["name"], payload,
        weight=item["weight"], kv=item["kv"], seed=seed,
    )
    for item, seed, payload in validation_records
]
validation_table = pd.DataFrame(validation_rows)
display(validation_table.sort_values(["trial", "seed"]))

### 16. Confirm the winner at 1,024-token prefill

In [ ]:
validation_summary = validation_table.groupby("trial").agg(
    mean_ppl=("ppl", "mean"),
    worst_relative_ppl=("relative_ppl", "max"),
    mean_cache_kl=("cache_kl", "mean"),
    worst_cache_kl=("cache_kl", "max"),
    mean_total_system_GB=("total_system_GB_at_prompt", "mean"),
).reset_index()
valid_release = validation_summary[
    (validation_summary["mean_ppl"] <= source_ppl * (1 + MEAN_PPL_GATE))
    & (validation_summary["worst_relative_ppl"] <= WORST_PPL_GATE)
    & (validation_summary["worst_cache_kl"] <= cache_kl_gate)
]
if valid_release.empty:
    print("No finalist passed every release gate; choosing the lowest mean joint loss for diagnosis.")
    validation_summary["fallback_score"] = (
        validation_summary["mean_ppl"] / source_ppl - 1
        + KV_KL_SCORE_WEIGHT * validation_summary["mean_cache_kl"]
    )
    winner_name = validation_summary.sort_values("fallback_score").iloc[0]["trial"]
else:
    winner_name = valid_release.sort_values("mean_total_system_GB").iloc[0]["trial"]
winner = next(item for item in release_finalists if item["name"] == winner_name)
print({"winner": winner_name})
display(validation_summary)

long_context_results = {}
if RUN_LONG_CONTEXT_CONFIRMATION:
    long_overrides = [
        "eval.perplexity=false", "eval.trajectory=false",
        "eval.kv_cache.batches=4", "eval.kv_cache.eval_offset_batches=4",
        "eval.kv_cache.prompt_len=1024", "eval.kv_cache.continuation_len=32",
        "eval.kv_cache.skip=4096",
    ]
    long_context_results["winner"] = run_trial(
        "long_context", f"{winner_name}__ctx1024",
        [*winner["overrides"], *long_overrides],
        seed=0, weight=winner["weight"], kv=winner["kv"],
    )
    source_long_overrides = [
        "patch.enabled=false", *KV_PROFILES[winner["kv"]],
        *COMMON_EVAL, *long_overrides,
    ]
    long_context_results["source_weights"] = run_trial(
        "long_context", f"source_weights__{winner['kv']}__ctx1024",
        source_long_overrides,
        seed=0, weight="source_w16", kv=winner["kv"],
    )
else:
    print("Long-context confirmation disabled.")

## Checks and results

### 17. Validate protocol invariants

In [ ]:
for key, payload in trial_records.items():
    config = payload["config"]
    cache_config = config.get("eval", {}).get("kv_cache")
    if not cache_config:
        continue
    dynamic = cache_config.get("dynamic") or {}
    frozen = cache_config.get("frozen_recipe") or []
    selection_batches = int(dynamic.get("selection_batches", 0))
    assert int(cache_config.get("eval_offset_batches", 0)) >= selection_batches
    metrics = payload["metrics"].get("kv_cache", {})
    assert metrics.get("prefill_key_nmse", 0) >= 0
    assert metrics.get("prefill_value_nmse", 0) >= 0
    if dynamic:
        assert metrics.get("dynamic", {}).get("target_reached"), key
    if frozen:
        assert metrics.get("frozen_recipe", {}).get("validated_layers") == len(frozen), key
print(f"Protocol checks passed for {len(trial_records)} notebook trial records.")

### 18. Persist tables, plots, summary, and an experiment-log entry

In [ ]:
import shutil

weight_screen_path = RESULT_ROOT / "weight_screen.csv"
joint_matrix_path = RESULT_ROOT / "joint_matrix.csv"
release_path = RESULT_ROOT / "release_validation.csv"
weight_screen.to_csv(weight_screen_path, index=False)
joint_matrix.to_csv(joint_matrix_path, index=False)
validation_table.to_csv(release_path, index=False)

winner_validation = validation_summary[
    validation_summary["trial"] == winner_name
].iloc[0]
long_summary = {}
for name, payload in long_context_results.items():
    cache = payload["metrics"]["kv_cache"]
    long_summary[name] = {
        "cache_kl": cache["mean_teacher_kl"],
        "cache_nll_delta": cache["nll_delta"],
        "cache_top1": cache["top1_agreement"],
        "effective_kv_bpv": cache["effective_kv_bpv"],
        "deployed_total_cache_bytes": cache["deployed_total_cache_bytes"],
        "prefill_key_nmse": cache["prefill_key_nmse"],
        "prefill_value_nmse": cache["prefill_value_nmse"],
    }

summary = {
    "git_sha": commit,
    "model": MODEL_ID,
    "source_ppl": source_ppl,
    "weight_finalists": weight_finalists,
    "joint_pareto_trials": joint_pareto["trial"].tolist(),
    "release_finalists": release_names,
    "winner": winner_name,
    "winner_mean_ppl": float(winner_validation["mean_ppl"]),
    "winner_worst_relative_ppl": float(winner_validation["worst_relative_ppl"]),
    "winner_mean_cache_kl": float(winner_validation["mean_cache_kl"]),
    "winner_worst_cache_kl": float(winner_validation["worst_cache_kl"]),
    "cache_kl_gate": float(cache_kl_gate),
    "frozen_kv_source": str(FROZEN_KV_SUMMARY_PATH),
    "frozen_kv_recommendation": frozen_kv_summary["recommendation"],
    "long_context": long_summary,
    "fallback_memory_warning": (
        "Packed bytes are logical deployment estimates; CUDA fallback VRAM is invalid."
    ),
}
summary_path = RESULT_ROOT / "joint_summary.json"
with summary_path.open("w") as handle:
    json.dump(summary, handle, indent=2)

log_lines = [
    f"## Qwen3.5-4B whole-system co-design ({commit[:12]})",
    "",
    f"- Source WikiText-2 PPL: {source_ppl:.4f}",
    f"- Weight finalists: {', '.join(weight_finalists)}",
    f"- Joint Pareto trials: {', '.join(joint_pareto['trial'].tolist())}",
    f"- Release winner: {winner_name}",
    f"- Winner mean PPL: {winner_validation['mean_ppl']:.4f}",
    f"- Winner worst relative PPL: {winner_validation['worst_relative_ppl']:+.2%}",
    f"- Winner mean/worst cache KL: {winner_validation['mean_cache_kl']:.6f} / {winner_validation['worst_cache_kl']:.6f}",
    f"- Frozen K/V candidate: {frozen_kv_summary['recommendation']} from {FROZEN_KV_SUMMARY_PATH}",
    "- All dynamic selection prefixes were disjoint from matched evaluation calls.",
    "- CUDA fallback measurements are quality-only; packed byte accounting is logical.",
]
log_path = RESULT_ROOT / "experiment_log_entry.md"
log_path.write_text("\n".join(log_lines) + "\n")

archive_base = RESULT_BASE / f"qwen35_joint_{commit[:12]}"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", RESULT_ROOT))
print(json.dumps(summary, indent=2))
print({
    "weight_screen": str(weight_screen_path),
    "joint_matrix": str(joint_matrix_path),
    "release_validation": str(release_path),
    "summary": str(summary_path),
    "experiment_log_entry": str(log_path),
    "archive": str(archive_path),
})

### 19. Download the compact result archive

In [ ]:
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(str(archive_path))
else:
    print(f"Results remain on Drive at {RESULT_ROOT}")

## Next steps

Send back `joint_summary.json`, `weight_screen.csv`, `joint_matrix.csv`,
`release_validation.csv`, and `experiment_log_entry.md`. A configuration
is publishable evidence only if it survives all three seeds, the matched
cache gate, the 1,024-token confirmation, and later native packed-runtime
benchmarks. The notebook intentionally keeps failed trials: negative
results are part of the algorithm record.